## **RQ4**

<!-- > _Em modelos globais para previsão multi-horizonte de poluentes, quando variáveis exógenas (meteorologia + co-poluentes) geram ganho real de desempenho — e esse ganho é explicável por atribuições consistentes e fisicamente plausíveis?_

* Avaliação empírica (não arquitetural) do impacto de exógenas;
* Horizontes de 7, 14 e 30 dias;
* Análise explícita de:
   * quando exógenas ajudam,
   * quando não ajudam,
   * possíveis razões físicas e estatísticas. -->

## Libraries

In [15]:
import os
os.environ['NIXTLA_ID_AS_COL'] = '1'

import optuna
import itertools
import shutil
import time
import functools
import gc
import requests
import pickle

import pandas as pd
import numpy as np
np.random.seed(1)

import plotly.graph_objects as go
import plotly.express as px
import plotly.subplots
import plotly.io as pio
from graphmodex import plotlymodex
pio.renderers.default = 'notebook'

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl

import joblib
import pickle
from IPython.display import clear_output

In [16]:
import neuralforecast
import mlforecast
import statsforecast
import utilsforecast
import coreforecast

from statsforecast import StatsForecast
from statsforecast.models import (
    Naive, SeasonalNaive,
    AutoARIMA, AutoCES, AutoETS, AutoTheta,
)

from mlforecast import MLForecast
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from mlforecast.lag_transforms import ExpandingMean, RollingMean
from mlforecast.target_transforms import Differences
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

from neuralforecast import NeuralForecast
from neuralforecast.models import (
    NBEATS, NHITS, NBEATSx,
    GRU, Informer, LSTM
)
from neuralforecast.losses.pytorch import MSE, SMAPE, MAE

from mlforecast.utils import PredictionIntervals

from pytorch_lightning import Trainer
trainer = Trainer(
    max_steps=4,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False
)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="optuna")


# ==================================================
# REPRODUCTIBILITY
# ==================================================
import random
import torch

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


### Results Storage

In [17]:
from pathlib import Path

BASE_RESULTS = Path("Results/RQ3")
BASE_RESULTS.mkdir(parents=True, exist_ok=True)

def save_results(df, model_family, pollutant, horizon_label, fold=None):

    out_dir = BASE_RESULTS / model_family
    out_dir.mkdir(parents=True, exist_ok=True)

    if fold is not None:
        fname = f"{pollutant}_{fold}_{horizon_label}.csv"
    else:
        fname = f"{pollutant}_{horizon_label}.csv"

    df.to_csv(out_dir / fname, index=False)

## Data

In [18]:
# ===============================
# DATA
# ===============================
df = pd.read_parquet(r'..\Data\CAMS\processed\eac4_era5_2010_2024_brasil_enhanced.parquet')


# ===============================
# POLLUTANTS
# ===============================
pm2p5 = (
    df
    .copy()
    .rename(columns={
        'pm2p5': 'y',
        'valid_time': 'ds'        
    })
    .query("ds >= '2010-01-01'")
)

go3 = (
    df
    .copy()
    .rename(columns={
        'go3': 'y',
        'valid_time': 'ds'        
    })
    .query("ds >= '2010-01-01'")
)

In [19]:
BASE_COLS = [ "ds", "unique_id", "y"]

O3_F1 = ["no2", "co"]
O3_F2 = ["u10", "v10", "wind_speed", "t2m", "d2m", "RH", "ssrd_flux", "blh"]
O3_F3 = O3_F1 + O3_F2

PM_F1 = ["so2", "no2", "co"]
PM_F2 = ["u10", "v10", "wind_speed", "t2m", "d2m", "RH", "ssrd_flux", "blh"]
PM_F3 = PM_F1 + PM_F2

In [20]:
pollutant_dict = {
    'go3': {
        'f0': {
            'df': go3[BASE_COLS].copy(),
            'exog_cols': []
        },
        'f1': {
            'df': go3[BASE_COLS + O3_F1].copy(),
            'exog_cols': O3_F1
        },
        'f2': {
            'df': go3[BASE_COLS + O3_F2].copy(),
            'exog_cols': O3_F2
        },
        'f3': {
            'df': go3[BASE_COLS + O3_F3].copy(),
            'exog_cols': O3_F3
        },
        'scaler': 1e8
    },

    'pm2p5': {
        'f0': {
            'df': pm2p5[BASE_COLS].copy(),
            'exog_cols': []
        },
        'f1': {
            'df': pm2p5[BASE_COLS + PM_F1].copy(),
            'exog_cols': PM_F1
        },
        'f2': {
            'df': pm2p5[BASE_COLS + PM_F2].copy(),
            'exog_cols': PM_F2
        },
        'f3': {
            'df': pm2p5[BASE_COLS + PM_F3].copy(),
            'exog_cols': PM_F3
        },
        'scaler': 1e9
    }
}

In [21]:
# Função para ajustar pela ordem inversa de grandeza de cada poluente
def adjust_by_inverse_magnitude(df, pollutants):
    for pollutant in pollutants:
        if pollutant in df.columns:
            # Calcular a mediana do poluente
            median_value = df[pollutant].median()
            # Calcular a ordem de grandeza (log base 10 da mediana)
            magnitude = np.floor(np.log10(abs(median_value)))
            # Multiplicar pela inversa da ordem de grandeza (10^(-magnitude))
            df[pollutant] *= 10 ** (-magnitude)
    return df

# Iterar sobre cada poluente no dicionário e ajustar os dados
for pollutant, data in pollutant_dict.items():
    for key, val in data.items():
        # Identificar os poluentes a serem ajustados (so2, no2, co)
        if isinstance(val, dict):
            df = val['df']
            exog_cols = val['exog_cols']
            # Ajustar as colunas de poluentes (se houver)
            df = adjust_by_inverse_magnitude(df, ['so2', 'no2', 'co'])

            # Atualizar o dicionário com o DataFrame ajustado
            pollutant_dict[pollutant][key]['df'] = df

# Resultado: dicionário de poluentes com os DataFrames ajustados

### Setup

In [22]:
# ===============================
# CONFIG
# ===============================
steps_per_day = 8
two_year_steps = 2 * 365 * steps_per_day
target_windows = 30

FREQ = '3h'
SEASON_LENGTH = 8 

experiments_dict = {
    '30 days': {
        'horizon': 8*30,
        'step_size': 8*30,
        'windows': two_year_steps // (8*30),  # 24
    },
    '14 days': {
        'horizon': 8*14,
        'step_size': max(8*14, two_year_steps // target_windows),
        'windows': target_windows,
    },
    '7 days': {
        'horizon': 8*7,
        'step_size': max(8*7, two_year_steps // target_windows),
        'windows': target_windows,
    },
}

## Merge

In [ ]:
BASE_RESULTS = Path(r"C:\Users\gustavo.filho\Documents\Python\Masters_New\Masters 26\Results\RQ4")

FULL_DIR = BASE_RESULTS / "full"
FULL_DIR.mkdir(parents=True, exist_ok=True)

families = ["ml", "dl"]
pollutants = ["go3", "pm2p5"]
folds = ["f0", "f1", "f2", "f3"]
horizons = ["7days", "14days", "30days"]


def build_full_results(pollutant, fold, horizon):

    dfs = []

    for family in families:

        fpath = BASE_RESULTS / family / f"{pollutant}_{fold}_{horizon}.csv"

        if fpath.exists():

            df = pd.read_csv(fpath)

            if "fit_time_seconds" in df.columns:
                df = df.drop(columns=["fit_time_seconds"])

            dfs.append(df)

        else:
            print(f"⚠️ Missing: {fpath}")

    if not dfs:
        return None

    # ordenar igualmente (sem fold)
    for i in range(len(dfs)):
        dfs[i] = dfs[i].sort_values(
            ["unique_id", "ds", "cutoff"]
        ).reset_index(drop=True)

    base = dfs[0][["unique_id", "ds", "cutoff", "y"]].copy()

    for df in dfs:

        model_cols = [
            c for c in df.columns
            if c not in ["unique_id", "ds", "cutoff", "y"]
        ]

        base = pd.concat(
            [base, df[model_cols]],
            axis=1
        )

    return base


for pollutant in pollutants:

    for fold in folds:

        for horizon in horizons:

            print(f"Building FULL | {pollutant} | {fold} | {horizon}")

            df_full = build_full_results(
                pollutant=pollutant,
                fold=fold,
                horizon=horizon
            )

            if df_full is None:
                continue

            df_full.to_csv(
                FULL_DIR / f"{pollutant}_{fold}_{horizon}.csv",
                index=False
            )

## **Estatísticas**

In [9]:
BASE_RESULTS = Path(r"C:\Users\gustavo.filho\Documents\Python\Masters_New\Masters 26\Results\RQ4")

FULL_DIR = BASE_RESULTS / "full"
FULL_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
def compute_forecast_metrics(y, yhat, y_base=None, y_lr=None):

    # ------------------------------------------
    # Convert to numeric safely
    # ------------------------------------------

    y = pd.to_numeric(pd.Series(y), errors="coerce").values
    yhat = pd.to_numeric(pd.Series(yhat), errors="coerce").values

    if y_base is not None:
        y_base = pd.to_numeric(pd.Series(y_base), errors="coerce").values

    if y_lr is not None:
        y_lr = pd.to_numeric(pd.Series(y_lr), errors="coerce").values

    # ------------------------------------------
    # Remove NaN rows
    # ------------------------------------------

    mask = ~np.isnan(y) & ~np.isnan(yhat)

    if mask.sum() == 0:
        return None

    y = y[mask]
    yhat = yhat[mask]

    if y_base is not None:
        y_base = y_base[mask]

    if y_lr is not None:
        y_lr = y_lr[mask]

    # ------------------------------------------
    # BASIC METRICS
    # ------------------------------------------

    mae = np.mean(np.abs(y - yhat))
    rmse = np.sqrt(np.mean((y - yhat) ** 2))

    denom = (np.abs(y) + np.abs(yhat)) / 2
    sm_mask = denom != 0
    smape = np.mean(np.abs(y[sm_mask] - yhat[sm_mask]) / denom[sm_mask]) if sm_mask.sum() > 0 else np.nan

    # ------------------------------------------
    # EXTREME THRESHOLDS
    # ------------------------------------------

    p95 = np.percentile(y, 95)
    p99 = np.percentile(y, 99)

    def cond_mae(a, b, thr):
        m = a >= thr
        return np.mean(np.abs(a[m] - b[m])) if m.sum() > 0 else np.nan

    def cond_bias(a, b, thr):
        m = a >= thr
        return np.mean(b[m] - a[m]) if m.sum() > 0 else np.nan

    mae_p95 = cond_mae(y, yhat, p95)
    mae_p99 = cond_mae(y, yhat, p99)

    bias_p95 = cond_bias(y, yhat, p95)
    bias_p99 = cond_bias(y, yhat, p99)

    # ------------------------------------------
    # TAIL ERROR RATIO
    # ------------------------------------------

    ter_p95 = mae_p95 / mae if mae > 0 else np.nan
    ter_p99 = mae_p99 / mae if mae > 0 else np.nan

    # ------------------------------------------
    # EVENT DETECTION
    # ------------------------------------------

    y_event = y >= p95
    yhat_event = yhat >= p95

    tp = np.sum(y_event & yhat_event)
    fp = np.sum(~y_event & yhat_event)
    fn = np.sum(y_event & ~yhat_event)
    tn = np.sum(~y_event & ~yhat_event)

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    if precision + recall == 0:
        f1 = np.nan
    else:
        f1 = 2 * precision * recall / (precision + recall)

    far = fp / (fp + tn) if (fp + tn) > 0 else np.nan

    # ------------------------------------------
    # SKILL FUNCTION
    # ------------------------------------------

    def skill(model_err, base_err):
        if base_err is None or base_err == 0 or np.isnan(base_err):
            return np.nan
        return 1 - model_err / base_err

    # ------------------------------------------
    # BASELINE SKILL
    # ------------------------------------------

    if y_base is not None:

        mae_base = np.mean(np.abs(y - y_base))
        rmse_base = np.sqrt(np.mean((y - y_base) ** 2))

        denom = (np.abs(y) + np.abs(y_base)) / 2
        m = denom != 0
        smape_base = np.mean(np.abs(y[m] - y_base[m]) / denom[m]) if m.sum() > 0 else np.nan

        mae_base_p95 = cond_mae(y, y_base, p95)
        mae_base_p99 = cond_mae(y, y_base, p99)

        skill_mae = skill(mae, mae_base)
        skill_rmse = skill(rmse, rmse_base)
        skill_smape = skill(smape, smape_base)

        skill_p95 = skill(mae_p95, mae_base_p95)
        skill_p99 = skill(mae_p99, mae_base_p99)

    else:

        skill_mae = skill_rmse = skill_smape = skill_p95 = skill_p99 = np.nan

    # ------------------------------------------
    # LINEAR BASELINE
    # ------------------------------------------

    if y_lr is not None:

        mae_lr = np.mean(np.abs(y - y_lr))
        rmse_lr = np.sqrt(np.mean((y - y_lr) ** 2))

        denom = (np.abs(y) + np.abs(y_lr)) / 2
        m = denom != 0
        smape_lr = np.mean(np.abs(y[m] - y_lr[m]) / denom[m]) if m.sum() > 0 else np.nan

        mae_lr_p95 = cond_mae(y, y_lr, p95)
        mae_lr_p99 = cond_mae(y, y_lr, p99)

        skill_mae_lr = skill(mae, mae_lr)
        skill_rmse_lr = skill(rmse, rmse_lr)
        skill_smape_lr = skill(smape, smape_lr)

        skill_p95_lr = skill(mae_p95, mae_lr_p95)
        skill_p99_lr = skill(mae_p99, mae_lr_p99)

    else:

        skill_mae_lr = skill_rmse_lr = skill_smape_lr = skill_p95_lr = skill_p99_lr = np.nan

    return {

        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": smape,

        "MAE_p95": mae_p95,
        "MAE_p99": mae_p99,

        "Bias_p95": bias_p95,
        "Bias_p99": bias_p99,

        "TER_p95": ter_p95,
        "TER_p99": ter_p99,

        "Precision_p95": precision,
        "Recall_p95": recall,
        "F1_p95": f1,
        "FalseAlarm_p95": far,

        "Skill_MAE": skill_mae,
        "Skill_RMSE": skill_rmse,
        "Skill_sMAPE": skill_smape,
        "Skill_p95": skill_p95,
        "Skill_p99": skill_p99,

        "Skill_MAE_LR": skill_mae_lr,
        "Skill_RMSE_LR": skill_rmse_lr,
        "Skill_sMAPE_LR": skill_smape_lr,
        "Skill_p95_LR": skill_p95_lr,
        "Skill_p99_LR": skill_p99_lr
    }


# ==================================================
# CONFIG
# ==================================================

RQ3_BASE = Path("Results/RQ3/full")

pollutants = ["go3", "pm2p5"]
horizons = ["7days", "14days", "30days"]
folds = ["f0", "f1", "f2", "f3"]

records = []


# ==================================================
# LOOP
# ==================================================

for pollutant in pollutants:

    for horizon in horizons:

        base_file = RQ3_BASE / f"{pollutant}_f0_{horizon}.csv"

        if not base_file.exists():
            print("Missing baseline:", base_file)
            continue

        df_base = pd.read_csv(base_file)

        df_base["ds"] = pd.to_datetime(df_base["ds"])
        df_base["cutoff"] = pd.to_datetime(df_base["cutoff"])

        model_cols = [
            c for c in df_base.columns
            if c not in ["unique_id", "ds", "cutoff", "y"]
        ]

        for fold in folds:

            file = RQ3_BASE / f"{pollutant}_{fold}_{horizon}.csv"

            if not file.exists():
                print("Missing:", file)
                continue

            print("Evaluating:", pollutant, fold, horizon)

            df = pd.read_csv(file)

            df["ds"] = pd.to_datetime(df["ds"])
            df["cutoff"] = pd.to_datetime(df["cutoff"])

            df = df.merge(
                df_base.drop(columns=["y"]),
                on=["unique_id", "ds", "cutoff"],
                suffixes=("", "_BASE")
            )

            for (uid, cutoff), df_fold in df.groupby(["unique_id", "cutoff"]):

                y = df_fold["y"].values

                for model in model_cols:

                    base_col = model + "_BASE"

                    if base_col not in df_fold.columns:
                        continue

                    yhat = df_fold[model].values
                    ybase = df_fold[base_col].values

                    if "LinearRegression" in df_fold.columns:
                        ylr = df_fold["LinearRegression"].values
                    else:
                        ylr = None

                    metrics = compute_forecast_metrics(
                        y,
                        yhat,
                        y_base=ybase,
                        y_lr=ylr
                    )

                    if metrics is None:
                        continue

                    records.append({

                        "pollutant": pollutant,
                        "fold": fold,
                        "horizon": horizon,
                        "unique_id": uid,
                        "cutoff": cutoff,
                        "model": model,

                        **metrics
                    })


# ==================================================
# SAVE
# ==================================================

metrics_df = pd.DataFrame(records)

print("Done.")

Evaluating: go3 f0 7days
Evaluating: go3 f1 7days
Evaluating: go3 f2 7days
Evaluating: go3 f3 7days
Evaluating: go3 f0 14days
Evaluating: go3 f1 14days
Evaluating: go3 f2 14days
Evaluating: go3 f3 14days
Evaluating: go3 f0 30days
Evaluating: go3 f1 30days
Evaluating: go3 f2 30days
Evaluating: go3 f3 30days
Evaluating: pm2p5 f0 7days
Evaluating: pm2p5 f1 7days
Evaluating: pm2p5 f2 7days
Evaluating: pm2p5 f3 7days
Evaluating: pm2p5 f0 14days
Evaluating: pm2p5 f1 14days
Evaluating: pm2p5 f2 14days
Evaluating: pm2p5 f3 14days
Evaluating: pm2p5 f0 30days
Evaluating: pm2p5 f1 30days
Evaluating: pm2p5 f2 30days
Evaluating: pm2p5 f3 30days
Done.


In [ ]:
metrics_df.to_csv(
    "Results/RQ4/metrics.csv",
    index=False
)

# Corrected Quantiles

In [30]:
def build_train_quantiles(df, pollutant):
    if pollutant == 'go3':
        df = go3.copy()
    else:
        df = pm2p5.copy()

    df_poll = df[df["ds"] <= "2022-12-31"]

    quantile_levels = np.arange(0.5, 1.0, 0.05)

    rows = []

    for uid, g in df_poll.groupby("unique_id"):

        qs = np.quantile(g["y"], quantile_levels)

        for q, val in zip(quantile_levels, qs):

            rows.append({
                "unique_id": uid,
                "quantile": q,
                "threshold": val
            })

    return pd.DataFrame(rows)


build_train_quantiles(df, 'go3')

,unique_id,quantile,threshold
0,0,0.50,3.860565e-08
1,0,0.55,3.969060e-08
2,0,0.60,4.083108e-08
3,0,0.65,4.198125e-08
4,0,0.70,4.316668e-08
...,...,...,...
2875,287,0.75,4.129273e-08
2876,287,0.80,4.409718e-08
2877,287,0.85,4.739405e-08
2878,287,0.90,5.176648e-08


In [28]:
import numpy as np
import pandas as pd
from pathlib import Path

# ==========================================================
# EXTREME METRICS
# ==========================================================

def compute_extreme_metrics(y, yhat, p95, p99, tolerance=0.05):

    y = pd.to_numeric(pd.Series(y), errors="coerce").values
    yhat = pd.to_numeric(pd.Series(yhat), errors="coerce").values

    mask = ~np.isnan(y) & ~np.isnan(yhat)

    if mask.sum() == 0:
        return None

    y = y[mask]
    yhat = yhat[mask]

    mae = np.mean(np.abs(y - yhat))

    # ------------------------------------------------------
    # conditional metrics
    # ------------------------------------------------------

    def cond_mae(a, b, thr):
        m = a >= thr
        return np.mean(np.abs(a[m] - b[m])) if m.sum() > 0 else np.nan

    def cond_rmse(a, b, thr):
        m = a >= thr
        return np.sqrt(np.mean((a[m] - b[m])**2)) if m.sum() > 0 else np.nan

    def cond_bias(a, b, thr):
        m = a >= thr
        return np.mean(b[m] - a[m]) if m.sum() > 0 else np.nan

    mae_p95 = cond_mae(y, yhat, p95)
    mae_p99 = cond_mae(y, yhat, p99)

    rmse_p95 = cond_rmse(y, yhat, p95)
    rmse_p99 = cond_rmse(y, yhat, p99)

    bias_p95 = cond_bias(y, yhat, p95)
    bias_p99 = cond_bias(y, yhat, p99)

    ter_p95 = mae_p95 / mae if mae > 0 else np.nan
    ter_p99 = mae_p99 / mae if mae > 0 else np.nan

    # ------------------------------------------------------
    # EVENT DETECTION
    # ------------------------------------------------------

    thr_detect = p95 * (1 - tolerance)

    y_event = y >= p95
    yhat_event = yhat >= thr_detect

    tp = np.sum(y_event & yhat_event)
    fp = np.sum(~y_event & yhat_event)
    fn = np.sum(y_event & ~yhat_event)
    tn = np.sum(~y_event & ~yhat_event)

    # ------------------------------------------------------
    # CLASSIC METRICS
    # ------------------------------------------------------

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else np.nan
    )

    far = fp / (tp + fp) if (tp + fp) > 0 else np.nan

    # ------------------------------------------------------
    # METEOROLOGICAL METRICS
    # ------------------------------------------------------

    pod = recall

    csi = tp / (tp + fn + fp) if (tp + fn + fp) > 0 else np.nan

    denom = (
        (tp + fn) * (fn + tn)
        + (tp + fp) * (fp + tn)
    )

    hss = (
        2 * (tp * tn - fn * fp) / denom
        if denom > 0 else np.nan
    )

    # Frequency Bias
    fb = (tp + fp) / (tp + fn) if (tp + fn) > 0 else np.nan

    # Brier Score
    brier = np.mean((y_event.astype(int) - yhat_event.astype(int))**2)

    return {

        "MAE": mae,

        "MAE_p95": mae_p95,
        "MAE_p99": mae_p99,

        "RMSE_p95": rmse_p95,
        "RMSE_p99": rmse_p99,

        "Bias_p95": bias_p95,
        "Bias_p99": bias_p99,

        "TER_p95": ter_p95,
        "TER_p99": ter_p99,

        "Precision_p95": precision,
        "Recall_p95": recall,
        "F1_p95": f1,

        "FAR_p95": far,
        "POD_p95": pod,
        "CSI_p95": csi,
        "HSS_p95": hss,

        "FB_p95": fb,
        "Brier_p95": brier,

        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    }


# ==========================================================
# TRAIN THRESHOLDS
# ==========================================================

def build_train_thresholds(df, pollutant):
    if pollutant == 'go3':
        df = go3.copy()
    else:
        df = pm2p5.copy()

    df_poll = df[df["ds"] <= "2022-12-31"]

    thresholds = (
        df_poll.groupby("unique_id")["y"]
        .agg(
            p95=lambda x: np.percentile(x, 95),
            p99=lambda x: np.percentile(x, 99)
        )
        .reset_index()
    )

    return thresholds


# ==========================================================
# TRAIN QUANTILES
# ==========================================================

def build_train_quantiles(df, pollutant):
    if pollutant == 'go3':
        df = go3.copy()
    else:
        df = pm2p5.copy()

    df_poll = df[df["ds"] <= "2022-12-31"]

    quantile_levels = np.arange(0.5, 1.0, 0.05)

    rows = []

    for uid, g in df_poll.groupby("unique_id"):

        qs = np.quantile(g["y"], quantile_levels)

        for q, val in zip(quantile_levels, qs):

            rows.append({
                "unique_id": uid,
                "quantile": q,
                "threshold": val
            })

    return pd.DataFrame(rows)


# ==========================================================
# CONFIG
# ==========================================================

RQ3_BASE = Path("Results/RQ3/full")

pollutants = ["go3", "pm2p5"]
horizons = ["7days", "14days", "30days"]
folds = ["f0", "f1", "f2", "f3"]

records = []
quantile_records = []

# df precisa conter TODOS os poluentes
# df = pd.read_parquet(...)

# ==========================================================
# MAIN LOOP
# ==========================================================

for pollutant in pollutants:

    print("Building climatology:", pollutant)

    train_thresholds = build_train_thresholds(df, pollutant)
    train_quantiles = build_train_quantiles(df, pollutant)

    for horizon in horizons:

        base_file = RQ3_BASE / f"{pollutant}_f0_{horizon}.csv"

        if not base_file.exists():
            continue

        df_base = pd.read_csv(base_file)

        df_base["ds"] = pd.to_datetime(df_base["ds"])
        df_base["cutoff"] = pd.to_datetime(df_base["cutoff"])

        model_cols = [
            c for c in df_base.columns
            if c not in ["unique_id", "ds", "cutoff", "y"]
        ]

        for fold in folds:

            file = RQ3_BASE / f"{pollutant}_{fold}_{horizon}.csv"

            if not file.exists():
                continue

            print("Evaluating:", pollutant, fold, horizon)

            df_pred = pd.read_csv(file)

            df_pred["ds"] = pd.to_datetime(df_pred["ds"])
            df_pred["cutoff"] = pd.to_datetime(df_pred["cutoff"])

            df_pred = df_pred.merge(
                train_thresholds,
                on="unique_id",
                how="left"
            )

            if df_pred["p95"].isna().any():
                print("WARNING: missing thresholds")

            for (uid, cutoff), df_fold in df_pred.groupby(["unique_id", "cutoff"]):

                y = df_fold["y"].values

                p95 = df_fold["p95"].iloc[0]
                p99 = df_fold["p99"].iloc[0]

                if np.isnan(p95):
                    continue

                lead_time = (df_fold["ds"] - df_fold["cutoff"]).dt.days.values

                for model in model_cols:

                    yhat = df_fold[model].values

                    metrics = compute_extreme_metrics(
                        y,
                        yhat,
                        p95,
                        p99,
                        tolerance=0.05
                    )

                    if metrics is None:
                        continue

                    records.append({

                        "pollutant": pollutant,
                        "fold": fold,
                        "horizon": horizon,
                        "unique_id": uid,
                        "cutoff": cutoff,

                        "lead_time_mean": np.mean(lead_time),

                        "model": model,

                        **metrics
                    })

                    # --------------------------------------------------
                    # ERROR VS QUANTILE
                    # --------------------------------------------------

                    q_df = train_quantiles[
                        train_quantiles["unique_id"] == uid
                    ]

                    for _, qrow in q_df.iterrows():

                        thr = qrow["threshold"]
                        q = qrow["quantile"]

                        mask = y >= thr

                        if mask.sum() == 0:
                            continue

                        mae_q = np.mean(np.abs(y[mask] - yhat[mask]))

                        quantile_records.append({

                            "pollutant": pollutant,
                            "fold": fold,
                            "horizon": horizon,
                            "unique_id": uid,
                            "cutoff": cutoff,

                            "model": model,

                            "quantile": q,
                            "MAE_quantile": mae_q
                        })


# ==========================================================
# FINAL DATAFRAMES
# ==========================================================

metrics_extreme = pd.DataFrame(records)
metrics_quantile = pd.DataFrame(quantile_records)

print("Extreme metrics shape:", metrics_extreme.shape)
print("Quantile metrics shape:", metrics_quantile.shape)


# ==========================================================
# SAVE RESULTS
# ==========================================================

metrics_extreme.to_csv("extreme_metrics_results.csv", index=False)
metrics_quantile.to_csv("quantile_error_results.csv", index=False)

print("Results saved.")

Building climatology: go3
Evaluating: go3 f0 7days
Evaluating: go3 f1 7days
Evaluating: go3 f2 7days
Evaluating: go3 f3 7days
Evaluating: go3 f0 14days
Evaluating: go3 f1 14days
Evaluating: go3 f2 14days
Evaluating: go3 f3 14days
Evaluating: go3 f0 30days
Evaluating: go3 f1 30days
Evaluating: go3 f2 30days
Evaluating: go3 f3 30days
Building climatology: pm2p5
Evaluating: pm2p5 f0 7days
Evaluating: pm2p5 f1 7days
Evaluating: pm2p5 f2 7days
Evaluating: pm2p5 f3 7days
Evaluating: pm2p5 f0 14days
Evaluating: pm2p5 f1 14days
Evaluating: pm2p5 f2 14days
Evaluating: pm2p5 f3 14days
Evaluating: pm2p5 f0 30days
Evaluating: pm2p5 f1 30days
Evaluating: pm2p5 f2 30days
Evaluating: pm2p5 f3 30days
Extreme metrics shape: (1741824, 29)
Quantile metrics shape: (15699816, 8)
Results saved.


In [29]:
metrics_extreme.to_csv("metrics_extreme.csv", index=False)
metrics_quantile.to_csv("metrics_quantile_curve.csv", index=False)

print("Done.")

Done.


# Full Database

In [ ]:
for poll_ in ['go3', 'pm2p5']:
    for horizon_ in ['7days', '14days', '30days']:

        f0 = pd.read_csv(rf'./Results/RQ3/full/{poll_}_f0_{horizon_}.csv')
        f3 = pd.read_csv(rf'./Results/RQ3/full/{poll_}_f3_{horizon_}.csv')

        fx = (
            f0.rename(columns={
                'NBEATSx-I': 'NBEATS-I',
                'NBEATSx-G': 'NBEATS-G',
            })
            [[
                'unique_id', 'ds', 'cutoff', 'y', 
                'LightGBM', 'GRU', 'LSTM',
                'LinearRegression', 
                'NHITS', 'NBEATS-I', 'NBEATS-G'
            ]]
            .merge(
                f3[[
                    'unique_id', 'ds', 'cutoff', 'y', 
                    'LightGBM', 'GRU', 'LSTM',
                    'LinearRegression', 
                    'NHITS', 'NBEATSx-I', 'NBEATSx-G'
                ]], 
                on=['unique_id', 'ds', 'cutoff', 'y'], 
                how='left', suffixes=('', 'x')
            )
        )

        fx.to_csv(rf'./Results/RQ4/full/{poll_}_fx_{horizon_}.csv', index=False)